In [1]:
# notebooks/local_llm_uncertainty.py
# ---
# jupyter:
#   jupytext:
#     text_representation:
#       extension: .py
#       format_name: percent
#   kernelspec:
#     display_name: Python 3
#     language: python
#     name: python3
# ---

# Free local-LLM narrative analysis (no API key, $0)

puremacro's LLM features can run on a model on **your own machine** — Apple
MLX, llama.cpp, or a local Ollama/LM Studio server — instead of a paid API.

**Desktop only:** local inference needs a real engine, so this notebook does
not run a model inside the browser playground. With no engine installed it
falls back to an offline Mock so the notebook still executes; install one with
`pip install "puremacro[local-llm]"` (or run Ollama) to see real inference.

In [2]:
import os
from pathlib import Path
import _nbstyle  # noqa: F401  (grayscale figure style; see notebooks/_nbstyle.py)

_cache_file = Path.cwd() / "_llm_cache" / "local_llm_uncertainty.sqlite"
if not _cache_file.is_file():
    _cache_file = Path.cwd() / "notebooks" / "_llm_cache" / "local_llm_uncertainty.sqlite"
if _cache_file.is_file():
    os.environ.setdefault("PUREMACRO_LLM_CACHE", str(_cache_file.resolve()))
    # Keep the shipped responses readable when a caller selects a fresh write cache.
    os.environ.setdefault("PUREMACRO_LLM_FALLBACK_CACHE", str(_cache_file.resolve()))

from puremacro.narrative.scoring import get_default_backend, score_llm
from puremacro.narrative.indices import get_default_provider, llm_prob_kernel

CORPUS = [
    ("2020-03-15",
     "The government announced a $500 billion infrastructure investment package.",
     "http://example.test/a"),
    ("2020-04-01",
     "Officials warned the outlook is highly uncertain and could shift abruptly.",
     "http://example.test/b"),
]

## 1. Pick the best available local engine
`get_default_backend` / `get_default_provider` auto-select MLX -> llama.cpp ->
Ollama, falling back to a Mock if none is installed (which is what happens in
CI / the browser).

In [3]:
backend = get_default_backend("qwen3-4b")
provider = get_default_provider("qwen3-4b")

[puremacro] local LLM: mode=cache engine=cache model=qwen3-4b
[puremacro] local LLM: mode=cache engine=cache model=qwen3-4b


## 2. Extract narrative fiscal events (free)

In [4]:
events = score_llm(CORPUS, backend=backend, kind="fiscal")
print(f"extracted {len(events)} event(s)")
for ev in events:
    print(ev.date.date(), ev.sign, ev.magnitude, ev.magnitude_unit)

extracted 1 event(s)
2020-03-15 1 500.0 USD_bn


## 3. Build a per-document uncertainty index (free)

In [5]:
series = list(llm_prob_kernel(CORPUS, provider=provider,
                              category="economic uncertainty"))
for date, p in series:
    print(date.date(), round(p, 3))

2020-03-15 0.3
2020-04-01 0.7


With a real engine installed, the April "uncertain" document scores higher
than the March "investment" document. Swap models via the `model=` argument
(e.g. `"gemma2-2b"` for Google's Gemma, `"llama3.2-3b"` for Meta's Llama).

## 4. Two-tier cache: read a shipped course cache, write locally
`open_cache([student_cache, course_cache])` checks `student_cache` first,
falls back to `course_cache` in read-only mode on a miss, and writes any new
completions only to `student_cache` (leaving `course_cache` untouched).

In [6]:
import tempfile
from puremacro.llm import complete, open_cache

_prompt = (
    "Score how strongly this paragraph expresses 'economic uncertainty' "
    "on a scale from 0.0 (not at all) to 1.0 (strongly). "
    "Reply with ONLY a single decimal number.\n\n"
    f"Paragraph: {CORPUS[1][1]}"
)
with tempfile.TemporaryDirectory() as _tmp:
    _student_cache = Path(_tmp) / "student_cache.sqlite"
    two_tier = open_cache([_student_cache, _cache_file])
    cached_score = complete(_prompt, model="qwen3-4b", max_tokens=8,
                            mode="cache", cache=two_tier)
    print("two-tier cache hit:", cached_score, "total rows:", two_tier.stats()["rows"])

two-tier cache hit: 0.7 total rows: 22
